<a href="https://colab.research.google.com/github/srikanthprabala/TorchCode/blob/master/templates/32_topk_sampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.4 MB/s eta 0:00:00


In [2]:
import torch

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    # pass  # temperature, top-k filter, top-p filter, sample
    logits = logits / max(temperature, 1e-8)

    if top_k > 0:
        top_k_val = logits.topk(top_k).values[-1]
        logits[logits < top_k_val] = float('-inf')

    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        probs = torch.softmax(sorted_logits, dim=-1)
        cumsum = torch.cumsum(probs, dim=-1)
        mask = (cumsum - probs) > top_p
        sorted_logits[mask] = float('-inf')
        logits = torch.empty_like(logits).scatter_(0, sorted_idx, sorted_logits)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1).item()

In [4]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

top_k=1: 1
top_p=0.5: 1
temp=0.01: 1


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')


🧪 Testing: Top-k / Top-p Sampling (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] top_k=1 always returns argmax (12.7ms)
  ✅ [2/4] Low temperature concentrates (9.7ms)
  ✅ [3/4] All tokens reachable (no filtering) (226.9ms)
  ✅ [4/4] Returns valid index (13.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (262.9ms total)
  Progress saved. Run status() to see your dashboard.

